# Slowing speech per phoneme with FastPitch

Companion notebook for **[Generate Slow, Don't Slow the Generation](https://ramwise.dev/blog/generate-slow-dont-slow-the-generation/)**.

To teach a word you slow it down — but a *slow word is not a uniformly slow word*. You stretch the vowels and **hold the stops**, because you cannot sustain a consonant burst. Doing that needs a model that exposes a **per-phoneme duration** knob.

[FastPitch](https://arxiv.org/abs/2006.06873) (Łańcucki, 2020) predicts a duration for every input symbol, and its synthesis takes a `pace` input — a multiplier *per token* (`pace < 1` = slower). That's the knob. This notebook shows the per-phoneme approach: a uniform slowdown (one number for everything) versus a class-aware one (hold vowels, keep stops crisp).

> **This needs a GPU.** On Google Colab: *Runtime → Change runtime type → T4 GPU*. It pulls NVIDIA NeMo's pretrained FastPitch + HiFi-GAN. Unlike the other pure-stdlib examples in this repo, this one is a teaching notebook meant to run on Colab — expect the usual version-pinning wrangle.

In [ ]:
# Colab: install NVIDIA NeMo (TTS). Takes a few minutes.
!pip install -q "nemo_toolkit[tts]"
# NeMo + Colab sometimes needs numba pinned:
!pip install -q "numba==0.60.0" 

In [ ]:
import torch
from nemo.collections.tts.models import FastPitchModel, HifiGanModel

device = "cuda" if torch.cuda.is_available() else "cpu"
fp = FastPitchModel.from_pretrained("tts_en_fastpitch").eval().to(device)   # acoustic model
voc = HifiGanModel.from_pretrained("tts_en_hifigan").eval().to(device)      # vocoder
tok = fp.tokenizer                                                          # ARPABET phoneme tokenizer

## The core: a per-phoneme `pace` vector

`generate_spectrogram` takes `pace` as either a single float (uniform) or a **tensor with one value per token**. To slow specific sounds, build a `pace` tensor of ones and overwrite the entries you want to lengthen. (NeMo inserts boundary tokens around the phonemes, so offset the phoneme index to the token index.)

In [ ]:
def synth(phonemes, pace_map=None):
    """phonemes: e.g. ["K","AE1","T"];  pace_map: {phoneme_index: pace}, pace < 1 = slower."""
    ids = tok.encode_from_g2p(phonemes)
    t = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    off = (len(ids) - len(phonemes)) // 2           # skip leading boundary token(s)
    pace = torch.ones(1, len(ids), device=device)   # 1.0 = natural speed, one value per token
    for i, pv in (pace_map or {}).items():
        j = off + i
        if 0 <= j < len(ids):
            pace[0, j] = pv
    with torch.no_grad():
        spec = fp.generate_spectrogram(tokens=t, pace=pace)
        audio = voc.convert_spectrogram_to_audio(spec=spec)
    return audio.cpu().numpy().reshape(-1)

## Uniform slowdown vs. class-aware slowdown

The naive way is one number for everything — `pace = 0.4` slows the whole word to 40% speed. It works up to a point, then the vowels drone and the stops smear. The phonics way slows each sound by its **class**: hold the vowels long, keep the stops at natural length.

In [ ]:
from IPython.display import Audio, display

STOPS = {"P", "B", "T", "D", "K", "G"}
is_vowel = lambda p: any(c.isdigit() for c in p)   # ARPABET vowels carry a stress digit, e.g. AE1

def class_pace(phonemes, slow=2.5):
    """Hold vowels, keep stops crisp, stretch continuants moderately."""
    pm = {}
    for i, p in enumerate(phonemes):
        if is_vowel(p):
            pm[i] = 1 / slow                     # vowels: fully slowed (held)
        elif p in STOPS:
            pm[i] = 1.0                          # stops: never stretched -- you can't hold a burst
        else:
            pm[i] = 1 / (1 + (slow - 1) * 0.5)   # continuants: about half as slowed
    return pm

cat = ["K", "AE1", "T"]

print("uniform 0.4x (everything slowed the same):")
display(Audio(synth(cat, {i: 0.4 for i in range(len(cat))}), rate=22050))

print("class-aware (hold the vowel, keep the /T/ crisp):")
display(Audio(synth(cat, class_pace(cat, slow=2.5)), rate=22050))

## Why this beats a single global knob

The browser version of this tool runs [Piper](https://github.com/rhasspy/piper) (a small VITS model, entirely client-side in WebAssembly — and surprisingly good). But Piper exposes exactly **one** timing knob: a single global `length_scale`. You can slow the whole word; you can't hold the vowel and keep the stop crisp in one pass. With only a scalar, class-aware timing has to be *faked* by synthesizing sounds separately and stitching them together — which is exactly why FastPitch's per-phoneme `pace` is the higher-fidelity path.

**The honest catch (model-agnostic).** No model, FastPitch included, renders a two-second held vowel gracefully — that "teaching register" is almost absent from normal-speech training data, so past a point any model decays into a frozen haze. Per-phoneme `pace` buys you more headroom, not infinite headroom.

The shipped app also tunes the per-class scales, and level-matches the quiet stops against the loud vowels, by ear — those constants are what turn "technically slowed" into "sounds like a teacher," and they're left out here on purpose. What's above is the general, teachable mechanism.

### References
- Łańcucki, *FastPitch: Parallel Text-to-speech with Pitch Prediction*, [arXiv:2006.06873](https://arxiv.org/abs/2006.06873) (ICASSP 2021).
- [NVIDIA NeMo — FastPitch finetuning tutorial](https://github.com/NVIDIA/NeMo/blob/main/tutorials/tts/FastPitch_Finetuning.ipynb).
- [Piper](https://github.com/rhasspy/piper) — the VITS model behind the browser version.